In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import numpy as np
import shapely

from gridr.core.grid import grid_commons
from gridr.core.grid import grid_rasterize

from notebook_utils import plot_convention_grid_mesh, mpl_plot_wrapper

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# Shared raster geometry used throughout the masking tutorial series.
shape, resolution = (6, 8), (1, 1)
origin_int, origin_half = (0., 0.), (0.5, 0.5)

# Grid-coordinate arrays for both conventions, used by plotting helpers.
cxx_int,  cyy_int  = grid_commons.grid_regular_coords_2d(shape, origin_int,  resolution, sparse=False)
cxx_half, cyy_half = grid_commons.grid_regular_coords_2d(shape, origin_half, resolution, sparse=False)

# `build_mask` with geometries

`grid_mask.build_mask` is the GridR API for assembling a binary validity mask from any combination of vector geometries and raster input masks. This notebook covers the geometry-only path: how to specify *valid* and *invalid* polygons through `geometry_pair`.

**What you'll learn**

- The `build_mask` signature and the `Validity` enum
- How to provide a *valid* geometry through `geometry_pair`
- How to provide an *invalid* geometry through `geometry_pair`
- How both interact when used together

## Setting things up

In [ ]:
from gridr.core.grid import grid_mask

### The `build_mask` signature

```python
def build_mask(
        shape: Tuple[int, int],
        resolution: Tuple[int, int],
        out: np.ndarray,
        geometry_origin: Optional[Tuple[float, float]] = None,
        geometry_pair: Optional[Tuple[Optional[GeometryType],
                Optional[GeometryType]]] = None,
        mask_in: Optional[np.ndarray] = None,
        mask_in_target_win: Optional[np.ndarray] = None,
        mask_in_resolution: Optional[Tuple[int, int]] = None,
        oversampling_dtype: Optional[np.dtype] = None,
        mask_in_binary_threshold: float = 0.999,
        rasterize_kwargs: Optional[Dict] = None,
        init_out: bool = False,
        ) -> Optional[np.ndarray]
```

From the docstring:

> *`build_mask` operates solely on raster data and does not perform I/O. It combines information from two distinct mask types to build a binary raster mask at a target resolution (currently only full resolution, i.e. `(1, 1)`, is implemented).*

### The `Validity` convention

```python
class Validity(IntEnum):
    INVALID = 0
    VALID   = 1
```

This is the convention every GridR mask must follow.

## The `geometry_pair` argument

`geometry_pair` is a length-2 sequence:

- index 0 -- the **valid** geometry (`GeometryType` or `None`),
- index 1 -- the **invalid** geometry (`GeometryType` or `None`).

`GeometryType` is defined in `grid_rasterize`:

```python
GeometryType = Union[
    shapely.geometry.Polygon,
    List[shapely.geometry.Polygon],
    shapely.geometry.MultiPolygon,
]
```

### Valid geometry only

With a single valid polygon, pixels inside the polygon are set to `Validity.VALID` and the rest to `Validity.INVALID`.

In [ ]:
rasterize_kwargs = {
    "alg": grid_rasterize.GridRasterizeAlg.RASTERIO_RASTERIZE,
    "kwargs_alg": {},
}

geometry_origin = (0.5, 0.5)
geometry_valid = shapely.geometry.Polygon([
    (3.5, 2.5),
    (6.5, 2.5),
    (6.5, 4.5),
    (3.5, 4.5),
])

raster = grid_mask.build_mask(
    shape            = shape,
    resolution       = (1, 1),
    out              = None,
    geometry_origin  = geometry_origin,
    geometry_pair    = [geometry_valid, None],
    rasterize_kwargs = rasterize_kwargs,
)

In [ ]:
# Default value-to-style map for binary mask plots
# (used by notebooks 003+ which display GridR Validity rasters).
mask_value_color_alpha_map = (
    (grid_mask.Validity.VALID,   "orange", 0.2),
    (grid_mask.Validity.INVALID, "blue",   0.2),
)

plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=geometry_valid, geometry_origin=geometry_origin,
    mask=raster, win=None,
    value_color_alpha_map=mask_value_color_alpha_map,
    prefix="build_mask_geometry_valid",
    title="Build mask with valid geometry",
)

### Invalid geometry only

A list of polygons can be passed to flag multiple disjoint invalid areas at once.

In [ ]:
geometry_invalid = [
    shapely.geometry.Polygon([
        (1.5, 1.5),
        (4.5, 1.5),
        (4.5, 2.5),
        (1.5, 2.5),
    ]),
    shapely.geometry.Polygon([
        (5.5, 3.5),
        (6.5, 3.5),
        (6.5, 4.5),
        (5.5, 4.5),
    ]),
]

raster = grid_mask.build_mask(
    shape            = shape,
    resolution       = (1, 1),
    out              = None,
    geometry_origin  = geometry_origin,
    geometry_pair    = [None, geometry_invalid],
    rasterize_kwargs = rasterize_kwargs,
)

In [ ]:
plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=geometry_invalid, geometry_origin=geometry_origin,
    mask=raster, win=None,
    value_color_alpha_map=mask_value_color_alpha_map,
    prefix="build_mask_geometry_invalid",
    title="Build mask with invalid geometry",
)

### Both at the same time

When both a valid and an invalid geometry are provided, internally two separate rasterisations are performed and merged with a binary `AND`: a pixel is valid in the output only if it is **valid in both masks**.

In practice this means the invalid geometry punches holes in the valid geometry.

In [ ]:
raster_geom = grid_mask.build_mask(
    shape            = shape,
    resolution       = (1, 1),
    out              = None,
    geometry_origin  = geometry_origin,
    geometry_pair    = [geometry_valid, geometry_invalid],
    rasterize_kwargs = rasterize_kwargs,
)
display(raster_geom)

In [ ]:
plot_convention_grid_mesh(
    shape, resolution, origin_int, cxx_int, cyy_int,
    geometry=None, geometry_origin=None,
    mask=raster_geom, win=None,
    value_color_alpha_map=mask_value_color_alpha_map,
    prefix="build_mask_geometry_valid_n_invalid",
    title="Build mask with valid and invalid geometries",
)

The next notebook covers the raster-input mode of `build_mask` -- supplying a pre-existing mask raster, possibly at a coarser resolution than the output.